# Activation Steering — **Phase 0**: 자기보고 도달성 체크

**질문**: α=0(스티어링 주입 없음)에서 **프롬프트 유도**(persona/grounded/fewshot)만으로 Likert 숫자 자기보고(digit)가 움직이나?
= 자기보고 readout 이 도달 가능한가. 다음 단계(v_selfreport 벡터)의 **전제 검증**.

- **추출/벡터 불필요** — 모델만 있으면 됨(extract/build/diagnostics 안 씀).
- 판정: `reachable`(digit 움직임→그 유도로 벡터) / `gated`(내부는 다른데 출력 고정→v_selfreport 동기) / `weak`(유도 강화).
- **-it(instruct) 모델 전용** (유도 prefix 는 chat 템플릿에서만 적용). 기본 `gemma-2-9b-it`@layer20.

## 1. GPU + 의존성 + HF 로그인

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q "transformers>=4.42" accelerate huggingface_hub openai python-dotenv bitsandbytes
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # gemma-2-9b-it 라이선스 수락 필요

## 2. 번들 업로드 + 모델 설정
Phase 0 는 가벼운 번들이면 충분: **`common.py` + `steering/*.py`(phase0_reachability·steer_eval·gemma_common) + `data/facets_en.json`**.
(pairs/추출 산출물 불필요.) zip 이름은 무관 — 첫 업로드 파일을 품.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()   # Phase 0 번들 zip
name = next(iter(up))
os.makedirs('/content/asteer', exist_ok=True)
with zipfile.ZipFile(name) as z: z.extractall('/content/asteer')
%cd /content/asteer

### 모델 설정 (여기만 바꾸면 됨)

In [ ]:
import os
os.environ['STEER_MODEL'] = 'google/gemma-2-9b-it'   # 2B로: google/gemma-2-2b-it
os.environ['STEER_LAYER'] = '20'                      # 내부측정 층. (2B면 이 줄 삭제→layer12)
# os.environ['STEER_4BIT'] = '1'                      # 무료 T4(16GB)면 주석 해제
print('MODEL =', os.environ['STEER_MODEL'], '| LAYER =', os.environ.get('STEER_LAYER'),
      '| 4BIT =', os.environ.get('STEER_4BIT','off'))

## 3. Phase 0 실행 (추출/벡터 불필요 — 모델만)

In [ ]:
!python steering/phase0_reachability.py --smoke   # 빠른 확인 ~1-2분 (neutral E_ft≈3.0 이어야 sanity)
!python steering/phase0_reachability.py           # 전체: persona/grounded/fewshot × ft+comp + 내부측정
# !python steering/phase0_reachability.py --layer-sweep   # (옵션) 내부측정 층별

## 4. 결과 보기 + 다운로드

In [ ]:
import json
r = json.load(open('artifacts/vectors/phase0_reachability.json'))
print('model', r['model'], '| layer', r['layer'], '| neutral E_ft =', r['neutral'].get('E_ft'), '(≈3.0 이어야 계측 sanity)')
for name, s in r['summary']['per_induction'].items():
    line = f"  [{name:8}] dE_ft={s.get('dE_ft')} dE_comp={s.get('dE_comp')} | dN_ft={s.get('dN_ft')} | spec={s.get('spec_ratio')} -> {s['diagnosis']}"
    if r.get('internal') and name in r['internal']['per_induction']:
        i = r['internal']['per_induction'][name]
        line += f"  (rel={i['rel_diff']} cos={i['cos_readout']})"
    print(line)
print('\n▶ v_selfreport 추천 유도:', r['summary']['recommended_for_vector'])
from google.colab import files; files.download('artifacts/vectors/phase0_reachability.json')

---
**판정 가이드**
- 어떤 유도든 **reachable**(|dE|≥0.5·특이성 OK) → 그 유도로 v_selfreport 대조 제작.
- 전부 **gated**(rel_diff 큼·cos≈0) → "내부는 다른데 출력만 3" 확증 = v_selfreport 정당화.
- **weak**(rel_diff 작음) → 유도 강화(더 긴 persona·예시↑·few-shot↑).

`phase0_reachability.json` 공유 바람 → Phase 1(v_selfreport 제작) 설계로.